# **DocStringer**  

A Python docstring generator built by finetuning Qwen2.5-Coder with QDoRA. Given an undocumented Python function, it automatically produces a clean Google-style docstring covering the description, arguments, and return value. Built with HuggingFace and PyTorch, and with a Gradio frontend for easy use.

-------------------------------------------------------------------------------


Initial Setup:

In [ ]:
# Only use the below code if on Google Colab, else comment out
from google.colab import drive
drive.mount('/content/gdrive')
!pip uninstall -y bitsandbytes accelerator peft transformers trl
!pip install -q torch==2.5.1+cu124 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q "transformers==4.47.0"
!pip install -q "peft==0.14.0"
!pip install -q "bitsandbytes==0.45.0" --extra-index-url https://vllm-wheels.pages.dev/high-efficiency
!pip install -q accelerate==1.2.1 trl==0.12.1 gradio datasets
import os
import sys
if os.path.exists('/usr/local/cuda/lib64'):
    os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

In [ ]:
"""
If you are running this notebook locally on your own GPU setup:
- Comment out the Google Colab section above.
- Uncomment the two lines below.
- Make sure to run 'pip install -r requirements.txt' in your local terminal.

OUTPUT_DIR = "./results/qwen-docstring-qdora"
"""

Imports:

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from peft import prepare_model_for_kbit_training,LoraConfig,PeftModel,get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

Loading the Pre-trained Qwen model and defining the QDora Config:

In [ ]:
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = prepare_model_for_kbit_training(model)

qdora_config = LoraConfig(
    r=8,
    lora_alpha = 16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
    use_dora=True #Making it so that its QDora instead of just QLora
)

model = get_peft_model(model, qdora_config)
model.print_trainable_parameters()

Preparing the Dataset:

In [ ]:
rawData = load_dataset("calum/the-stack-smol-python-docstrings", split="train")
print(rawData)
print("---")
# Look at a few examples
for i in range(3):
    print(f"=== Example {i+1} ===")
    print(rawData[i])
    print()

In [ ]:
#Utilized Claude to generate a function for limiting the dataset to only Google-style docstrings (to exclude Sphinx style ones)
def isGoogleStyle(docstring):
    if not docstring or len(docstring.strip()) < 20:
        return False
    google_markers = ["Args:", "Returns:", "Raises:", "Example:", "Note:"]
    sphinx_markers = [":param", ":type", ":return:", ":rtype:"]
    #Exclude Sphinx style
    if any(marker in docstring for marker in sphinx_markers):
        return False

    return any(marker in docstring for marker in google_markers)

def isQualityExample(example):
    docstring = example["docstring"]
    body = example["body_without_docstring"]
    if not isGoogleStyle(docstring):
        return False
    #Function body shouldn't be too short nor too long
    if len(body.strip()) < 20:
        return False
    if len(docstring) > 1000:
        return False
    return True

filteredData = rawData.filter(isQualityExample)
print(f"Original dataset size: {len(rawData)}")
print(f"Filtered dataset size: {len(filteredData)}")
print(f"Kept {len(filteredData)/len(rawData)*100:.1f}% of examples")

Transforming the dataset examples into Qwen prompts:

In [ ]:
from trl import DataCollatorForCompletionOnlyLM

prompt = "You are an expert Python developer. Your task is to generate a clean, complete and accurate Google-style docstring for the given function. Your response must contain a concise description, clearly documented 'Args:' and 'Returns:' sections if applicable."
def formatPrompt(example):
  #Building the ChatML Structure
  messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Function to document:\n{example['body_without_docstring']}"},
        {"role": "assistant", "content": f"\"\"\"\n{example['docstring'].strip()}\n\"\"\""} #We need to ignore the initial and last """
  ]

  #Convert structured list into raw ChatML formatted tokens
  text = tokenizer.apply_chat_template(messages, tokenize=False)
  return {"text": text}

formattedData = filteredData.map(formatPrompt)
print(formattedData[0])


In [ ]:
responseTemplate = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template=responseTemplate, tokenizer=tokenizer)
print("Prompt Formatting complete")
print(formattedData[0]['text'][:450])

Setting the Training Arguments:

In [ ]:
#Doing a 90% 10% train-test split
splitData = formattedData.train_test_split(test_size=0.1, seed=42)
trainData = splitData["train"]
valData = splitData["test"]

print(f"Training examples: {len(trainData)}")
print(f"Validation examples: {len(valData)}")

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
#Change the below output folder path depending on if you're local or in Colab
output_dir = "/content/gdrive/MyDrive/qwen-docstring-qdora"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=10,
    num_train_epochs=1,
    max_steps=100, #Can comment this out when running for final deploymet
    #fp16=True, # Toggle to fp16=True if on T4 GPU else bf16=True
    bf16=True,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=50,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    push_to_hub=False,
    report_to="none"
)

Fine Tuning the Model:

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=trainData,
    eval_dataset=valData,
    dataset_text_field="text",
    max_seq_length=512,
    data_collator=collator,
)

print("Starting Fine-Tuning...")
trainer.train()
trainer.save_model(output_dir)
print("Fine-Tuning Complete!")


Merging the QDora weights back into the base model:

In [ ]:
from peft import PeftModel

print("Merging QDoRA adapters with the base model...")
# Load the saved adapters from Google Drive over your base model
inference_model = PeftModel.from_pretrained(model, output_dir)
inference_model.eval()  # Set to evaluation mode to disable dropout

def generate_docstring(raw_code: str) -> str:
    """Formats raw code into the ChatML template and streams the generated docstring."""
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Function to document:\n{raw_code}"}
    ]

    # add_generation_prompt=True forces the model to switch into assistant generation mode
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = inference_model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,  # Keep temperature low for deterministic code structure
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only the newly generated tokens
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Test Sample
test_code = """
def process_user_records(users, target_status):
    output = []
    for user in users:
        if user.get('status') == target_status:
            output.append(user['id'])
    return output
"""

print("\n--- Testing Model Inference ---")
print(generate_docstring(test_code))

Deploying the Frontend:

In [ ]:
import gradio as gr

def ui_wrapper(raw_code):
    if not raw_code.strip():
        return "Please paste a valid Python function definition."
    try:
        return generate_docstring(raw_code)
    except Exception as e:
        return f"An error occurred during generation: {str(e)}"

with gr.Blocks(title="DocStringer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🖥️✏️ **DocStringer**")
    gr.Markdown(
        "Paste a Python function on the left. The fine-tuned Qwen2.5-Coder model "
        "will automatically generate a structured Google-style docstring on the right."
    )

    with gr.Row():
        with gr.Column():
            input_component = gr.Textbox(
                label="Your Python Code:",
                lines=12,
                placeholder="def calculate_metrics(data):\n    ..."
            )

        with gr.Column():
            output_component = gr.Textbox(
                label="Generated Google-Style Docstring:",
                lines=12,
                interactive=False
            )
    with gr.Row():
        gr.Markdown()
        submit_btn = gr.Button("Generate", variant="primary", scale=1)
        gr.Markdown()

    submit_btn.click(fn=ui_wrapper, inputs=input_component, outputs=output_component)

#Launch the server (setting share=True provides a public link valid for 72 hours)
demo.launch(share=True, debug=True)

# References
-------------------------------------------------------------------------------
Tutorials and Original References
* https://www.geeksforgeeks.org/nlp/fine-tuning-large-language-models-llms-using-qlora/
* https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct
* https://huggingface.co/docs/peft/package_reference/lora
* https://claude.ai/
* https://huggingface.co/docs/trl/v0.9.6/en/sft_trainer#train-on-completions-only
* https://huggingface.co/datasets/calum/the-stack-smol-python-docstrings

Inference & Adapter Lifecycle
* Hugging Face PEFT Causal LM Inference Guide:
  https://huggingface.co/docs/peft/en/task_guides/causal_lm_adapter
* Hugging Face Generation Parameters & Strategies:
  https://huggingface.co/docs/transformers/main_classes/text_generation

UI & Deployment
* Gradio Blocks Layout Component Documentation:
  https://www.gradio.app/docs/gradio/blocks
* Hugging Face Spaces Hosting Overview:
  https://huggingface.co/docs/hub/spaces-overview